In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from shapely.geometry import Point, Polygon
from scipy.spatial import ConvexHull


In [2]:
excel_file_path0 = 'C:/Users/weisz/OneDrive/Documents/中信兄弟/程式碼/Pitch Value/RE288.xlsx'
sheet_name = 'Sheet1'
RE = pd.read_excel(excel_file_path0, sheet_name=sheet_name)

In [3]:
excel_file_path1 = 'C:/Users/weisz/OneDrive/Documents/中信兄弟/Season_data/2024_non_trackman.xlsx'
excel_file_path2 = 'C:/Users/weisz/OneDrive/Documents/中信兄弟/Season_data/2023_non_trackman.xlsx'
excel_file_path3 = 'C:/Users/weisz/OneDrive/Documents/中信兄弟/Season_data/2025_non_trackman.xlsx'
excel_file_path4 = 'C:/Users/weisz/OneDrive/Documents/中信兄弟/Season_data/2026_non_trackman.xlsx'
sheet_name = 'Sheet1'

In [4]:
data2024 = pd.read_excel(excel_file_path1, sheet_name=sheet_name)
data2023 = pd.read_excel(excel_file_path2, sheet_name=sheet_name)
data2025 = pd.read_excel(excel_file_path3, sheet_name=sheet_name)
data2026 = pd.read_excel(excel_file_path4, sheet_name=sheet_name)

In [5]:
col = ['LocalDateTime','GameID','umpireInChief','date','stadium','bHand','pHand','strikes','balls','outs','side','inning','coordX','coordY','pType','call','result','bName','bTeamId', 'pName','pTeamId', 'gameId', 'paRound', 'paOrder',"runsOnPlay",'firstId','secondId','thirdId','InducedVertBreak']

In [6]:
data2024= data2024[col]
data2023= data2023[col]
data2025= data2025[col]
data2026= data2026[col]

In [7]:
df = pd.concat([data2024, data2025,data2023,data2026], ignore_index=True)

In [8]:
df = df.dropna(subset=['GameID'])

In [9]:
name_mapping = {
    '65630ee55559b6d7a6d5d200':'兄弟',
    '65630ee95559b6d7a6d5d201':'味全', 
    '65630eed5559b6d7a6d5d202':'樂天', 
    '65630ef25559b6d7a6d5d203':'統一',
    '65630ef65559b6d7a6d5d204':'富邦',
    '6569a9ddaf62a48d3f94b422':'台鋼',
    }

# 新增一列"球員"，根据"Pitcher"列的值匹配中文名
df['進攻球隊'] = df['bTeamId'].map(name_mapping)

In [10]:
name_mapping = {
    '65630ee55559b6d7a6d5d200':'兄弟',
    '65630ee95559b6d7a6d5d201':'味全', 
    '65630eed5559b6d7a6d5d202':'樂天', 
    '65630ef25559b6d7a6d5d203':'統一',
    '65630ef65559b6d7a6d5d204':'富邦',
    '6569a9ddaf62a48d3f94b422':'台鋼',
    }

# 新增一列"球員"，根据"Pitcher"列的值匹配中文名
df['防守球隊'] = df['pTeamId'].map(name_mapping)

In [11]:
df.loc[:, 'Year'] = df['date'].str[:4]
#df.loc[:, 'Year'] = df['date'].astype(str).str[:4]
#df.loc[:, 'day'] = df['date'].str[:10]

In [12]:
df = df[~df['result'].isin(['GO','2B', '1B', 'FO', 'SH', 'E', 'GIDP', 'FC','HBP', 'HR', '3B', 'IBB', 'SF', 'IGNORE', 'DP', 'D3S', 'ID', 'IH'])]
df = df[~df['call'].isin(['F', 'SW', 'CS', 'FT_MISS', 'FOUL_BUNT','FT','TRY_BUNT', 'H', 'BUNT'])]
df= df.dropna(subset=['call'])

In [13]:
name_mapping = {
    'FF':'直球', 
    'CH':'變速球', 
    'FC':'卡特球', 
    'CU':'曲球', 
    'FO':'指叉球',
    'SL':'滑球', 
    'SI':'伸卡球',
    'KN':'彈指曲球', 
    'EP':'小便球'
}

# 新增一列"球員"，根据"Pitcher"列的值匹配中文名
df['球種'] = df['pType'].map(name_mapping)

In [14]:
feild_items = df['stadium'].unique()
feild_items

array(['臺北市立天母棒球場 Tianmu Baseball Stadium',
       '臺中洲際棒球場 \tTaichung Intercontinental Baseball Stadium',
       '斗六棒球場 \tDouliu Baseball Stadium',
       '臺中洲際棒球場 Taichung Intercontinental Baseball Stadium', '新北市立新莊棒球場',
       '臺北市立天母棒球場', '新北市立新莊棒球場 Xinzhuang Baseball Stadium', '臺北大巨蛋',
       '臺北大巨蛋 Taipei Dome', nan, '斗六棒球場', '臺中洲際棒球場', '亞太國際棒球訓練中心-主球場',
       '樂天桃園棒球場', '澄清湖棒球場', '亞太成棒主球場',
       '樂天桃園棒球場 Rakuten Taoyuan Baseball Stadium'], dtype=object)

In [15]:
# 1. 定義好球帶邊界
STRIKE_ZONE_LEFT = -50
STRIKE_ZONE_RIGHT = 50
STRIKE_ZONE_BOTTOM = -50
STRIKE_ZONE_TOP = 50

def in_strike_zone(x, y):
    # 使用 <= 與 >= 已經完整包含了「碰到線也算」的邏輯
    return (STRIKE_ZONE_LEFT <= x <= STRIKE_ZONE_RIGHT) and \
           (STRIKE_ZONE_BOTTOM <= y <= STRIKE_ZONE_TOP)

# 2. 判斷是否為規則上的好球 (Rulebook Strike)
df["rulebook_strike"] = df.apply(lambda row: in_strike_zone(row["coordX"], row["coordY"]), axis=1)

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 68475 entries, 1417 to 321039
Data columns (total 34 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   LocalDateTime     68475 non-null  object 
 1   GameID            68475 non-null  object 
 2   umpireInChief     68046 non-null  object 
 3   date              68475 non-null  object 
 4   stadium           68254 non-null  object 
 5   bHand             68475 non-null  object 
 6   pHand             68475 non-null  object 
 7   strikes           68475 non-null  int64  
 8   balls             68475 non-null  int64  
 9   outs              68475 non-null  int64  
 10  side              68475 non-null  object 
 11  inning            68475 non-null  int64  
 12  coordX            68474 non-null  float64
 13  coordY            68474 non-null  float64
 14  pType             68475 non-null  object 
 15  call              68475 non-null  object 
 16  result            3923 non-null   object 

In [17]:
RE

,Unnamed: 0,壘包狀態,壞球,好球,出局數,出現次數,總得分,Expected_Runs,比例,加權得分期望值
0,0,0,0,0,0,15553,9684,0.622645,0.045999,0.028641
1,1,0,0,0,1,10686,5897,0.551844,0.031604,0.017441
2,2,0,0,0,2,8220,4396,0.534793,0.024311,0.013001
3,4,0,0,1,0,9515,5892,0.619233,0.028141,0.017426
4,5,0,0,1,1,6461,3603,0.557654,0.019109,0.010656
...,...,...,...,...,...,...,...,...,...,...
283,294,7,3,1,1,114,169,1.482456,0.000337,0.000500
284,295,7,3,1,2,120,180,1.500000,0.000355,0.000532
285,296,7,3,2,0,93,145,1.559140,0.000275,0.000429
286,297,7,3,2,1,238,346,1.453782,0.000704,0.001023


In [18]:
df["is_correct"] = df.apply(
    lambda row: (row["call"] == "S" and row["rulebook_strike"]) or 
                (row["call"] == "B" and not row["rulebook_strike"]),
    axis=1
)

accuracy_by_umpire = df.groupby("umpireInChief")["is_correct"].mean().reset_index()
accuracy_by_umpire.rename(columns={"is_correct":"accuracy"}, inplace=True)
accuracy_by_umpire['accuracy']=accuracy_by_umpire['accuracy']*100
accuracy_by_umpire

,umpireInChief,accuracy
0,劉世偉,90.240642
1,吳家維,90.624241
2,尤志欽,90.807752
3,張展榮,90.406355
4,彭楚雲,90.520694
5,木內九二生,89.917127
6,林金達,89.507299
7,楊崇煇,90.053232
8,江春緯,89.661127
9,王俊宏,89.906542


## 裁判分數計算


In [19]:
df_ump = df.copy()

In [20]:
def base_state(row):
    state = 0
    if not pd.isna(row['firstId']):
        state += 1
    if not pd.isna(row['secondId']):
        state += 2
    if not pd.isna(row['thirdId']):
        state += 4
    return state

mask2 = df_ump['Year'].isin(['2023', '2024', '2025','2026'])
df_ump.loc[mask2, '壘包狀態編號'] = df_ump.loc[mask2].apply(base_state, axis=1)
df_ump.loc[mask2, '壘包狀態編號'] = df_ump.loc[mask2, '壘包狀態編號'].fillna(0)

df_ump['PitchCount'] = df_ump.groupby(['gameId', 'pName']).cumcount() + 1

In [21]:
name_mapping = {
    0:'無人在壘',
    1:'一壘有人', 
    2:'二壘有人',
    3:'一二壘有人',
    4:'三壘有人',
    5:'一三壘有人',
    6:'二三壘有人',
    7:'滿壘'
}

# 新增一列"球員"，根据"Pitcher"列的值匹配中文名
df_ump['壘包狀態'] = df_ump['壘包狀態編號'].map(name_mapping)

In [22]:
df_ump['PitchofPA'] = df_ump.groupby(['bName', 'pName', 'date', 'gameId', 'paRound', 'paOrder']).cumcount() + 1

In [23]:
# 1. 原始排序邏輯
df_ump.sort_values(by=['date','inning','gameId', 'paRound', 'paOrder','pName', 'bName', 'PitchofPA'], inplace=True)

# 2. 修改後的 get_expected_runs (加入空值檢查)
def get_expected_runs(row):
    base_type = row["壘包狀態編號"]
    outs = row["outs"]
    balls = row["balls"]
    strikes = row["strikes"]
    
    matching_row = RE[(RE["壘包狀態"] == base_type) & 
                      (RE["出局數"] == outs) & 
                      (RE["壞球"] == balls) & 
                      (RE["好球"] == strikes)]
    
    if not matching_row.empty:
        # 確保取出的值不是 None
        val = matching_row.iloc[0]["Expected_Runs"]
        return val if pd.notna(val) else 0
    else:
        return 0 # 找不到對照時回傳 0，避免 None 導致後續崩潰

# 執行計算
df_ump["Expected_Runs"] = df_ump.apply(get_expected_runs, axis=1)

# 3. 執行位移並正確賦值 (避免使用可能失效的 inplace=True)
df_ump["Next_Expected_Runs"] = df_ump.groupby(["date", "gameId", "inning", "side"])["Expected_Runs"].shift(-1)
df_ump["Next_Expected_Runs"] = df_ump["Next_Expected_Runs"].fillna(0)

In [24]:
# 1. 計算該球造成的期望值變動 (Expected Run Value Change)
df_ump['RE_change'] = df_ump['Next_Expected_Runs'] - df_ump['Expected_Runs']

# 2. 定義判別函數
def label_beneficiary(row):
    # 如果判決正確，通常視為公平競爭，但在進階分析中，
    # 若要看「判決結果」對誰有利，不論正確與否，逻辑如下：
    
    # 這裡我們專注於「判決後的結果」：
    # 如果 RE 增加，代表進攻方處境變好 -> 進攻球隊得利
    if row['RE_change'] > 0:
        return row['進攻球隊']
    # 如果 RE 減少，代表進攻方處境變差（好球數增加） -> 防守球隊得利
    elif row['RE_change'] < 0:
        return row['防守球隊']
    else:
        return '無影響'

# 3. 標記受益球隊
df_ump['benefited_team'] = df_ump.apply(label_beneficiary, axis=1)

# 4. 如果只想看「誤判」帶來的得利 (Umpire Error Impact)
def label_missed_call_benefit(row):
    if row['is_correct']:
        return '判決正確'
    
    # 誤判情況下：
    # RE 增加（例如好球誤判為壞球）：進攻方白賺了期望值
    if row['RE_change'] > 0:
        return f"{row['進攻球隊']}"
    # RE 減少（例如壞球誤判為好球）：防守方白賺了期望值
    elif row['RE_change'] < 0:
        return f"{row['防守球隊']}"
    return '誤判-無影響'

df_ump['ump_error_benefit'] = df_ump.apply(label_missed_call_benefit, axis=1)

# 4. 如果只想看「誤判」帶來的得利 (Umpire Error Impact)
def label_missed_call_benefit2(row):
    if row['is_correct']:
        return '判決正確'
    
    # 誤判情況下：
    # RE 增加（例如好球誤判為壞球）：進攻方白賺了期望值
    if row['RE_change'] > 0:
        return '進攻球隊'
    # RE 減少（例如壞球誤判為好球）：防守方白賺了期望值
    elif row['RE_change'] < 0:
        return '防守球隊'
    return '誤判-無影響'

df_ump['ump_error_benefit_team'] = df_ump.apply(label_missed_call_benefit2, axis=1)

df_ump

,LocalDateTime,GameID,umpireInChief,date,stadium,bHand,pHand,strikes,balls,outs,...,壘包狀態編號,PitchCount,壘包狀態,PitchofPA,Expected_Runs,Next_Expected_Runs,RE_change,benefited_team,ump_error_benefit,ump_error_benefit_team
211609,'2023-04-01T18:16:59.1500+08:00,20230401-Brothers-2,蘇建文,2023-04-01T09:21:00.000Z,臺中洲際棒球場,L,L,0,0,0,...,0.0,1,無人在壘,1,0.622645,0.617490,-0.005155,樂天,判決正確,判決正確
211612,'2023-04-01T18:17:44.9970+08:00,20230401-Brothers-2,蘇建文,2023-04-01T09:21:00.000Z,臺中洲際棒球場,L,L,2,1,0,...,0.0,2,無人在壘,2,0.617490,0.534793,-0.082697,樂天,判決正確,判決正確
211599,'2023-04-01T18:10:19.4460+08:00,20230401-Brothers-2,蘇建文,2023-04-01T09:21:00.000Z,臺中洲際棒球場,L,L,0,0,0,...,0.0,1,無人在壘,1,0.622645,0.654981,0.032335,樂天,判決正確,判決正確
211600,'2023-04-01T18:10:30.9980+08:00,20230401-Brothers-2,蘇建文,2023-04-01T09:21:00.000Z,臺中洲際棒球場,L,L,0,1,0,...,0.0,2,無人在壘,2,0.654981,0.669778,0.014797,樂天,樂天,進攻球隊
211601,'2023-04-01T18:10:44.8330+08:00,20230401-Brothers-2,蘇建文,2023-04-01T09:21:00.000Z,臺中洲際棒球場,L,L,0,2,0,...,0.0,3,無人在壘,3,0.669778,0.803063,0.133286,樂天,判決正確,判決正確
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
300020,'2026-05-06T21:32:19.7460+08:00,20260506-Guardians-1,陳乃瑞,2026-05-06T10:35:00.000Z,新北市立新莊棒球場,L,L,0,0,1,...,3.0,12,一二壘有人,1,1.092979,1.024454,-0.068525,富邦,判決正確,判決正確
300021,'2026-05-06T21:32:37.2060+08:00,20260506-Guardians-1,陳乃瑞,2026-05-06T10:35:00.000Z,新北市立新莊棒球場,L,L,1,0,1,...,3.0,13,一二壘有人,2,1.024454,1.140546,0.116092,樂天,判決正確,判決正確
300022,'2026-05-06T21:32:59.5880+08:00,20260506-Guardians-1,陳乃瑞,2026-05-06T10:35:00.000Z,新北市立新莊棒球場,L,L,1,1,1,...,3.0,14,一二壘有人,3,1.140546,1.061864,-0.078682,富邦,富邦,防守球隊
300024,'2026-05-06T21:34:15.8650+08:00,20260506-Guardians-1,陳乃瑞,2026-05-06T10:35:00.000Z,新北市立新莊棒球場,L,L,0,0,2,...,5.0,15,一三壘有人,1,1.061864,1.007825,-0.054039,富邦,判決正確,判決正確


In [25]:
df_ump.info()

<class 'pandas.core.frame.DataFrame'>
Index: 68475 entries, 211609 to 300025
Data columns (total 45 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   LocalDateTime           68475 non-null  object 
 1   GameID                  68475 non-null  object 
 2   umpireInChief           68046 non-null  object 
 3   date                    68475 non-null  object 
 4   stadium                 68254 non-null  object 
 5   bHand                   68475 non-null  object 
 6   pHand                   68475 non-null  object 
 7   strikes                 68475 non-null  int64  
 8   balls                   68475 non-null  int64  
 9   outs                    68475 non-null  int64  
 10  side                    68475 non-null  object 
 11  inning                  68475 non-null  int64  
 12  coordX                  68474 non-null  float64
 13  coordY                  68474 non-null  float64
 14  pType                   68475 non-nul

In [26]:
#df

# 一致性計算 繪圖

## 每一位裁判(2023~)

In [27]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from shapely.geometry import Point, Polygon
from scipy.stats import gaussian_kde
from scipy.spatial import ConvexHull

# --- 1. 環境與好球帶設定 ---
STRIKE_ZONE_LEFT, STRIKE_ZONE_RIGHT = -50, 50
STRIKE_ZONE_BOTTOM, STRIKE_ZONE_TOP = -50, 50

TRUE_ZONE_POLY = Polygon([
    (STRIKE_ZONE_LEFT, STRIKE_ZONE_BOTTOM), (STRIKE_ZONE_RIGHT, STRIKE_ZONE_BOTTOM),
    (STRIKE_ZONE_RIGHT, STRIKE_ZONE_TOP), (STRIKE_ZONE_LEFT, STRIKE_ZONE_TOP)
])

# 中文字體設定
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'Arial Unicode MS', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False

# --- 2. 核心計算函數 ---

def build_umpire_zone(df_subset, bandwidth=12, threshold=0.35):
    """建立裁判感官好球帶 (KDE)"""
    strikes = df_subset[df_subset["call"] == "S"][["coordX", "coordY"]].dropna().values
    if len(strikes) < 4: return None 
    try:
        kde = gaussian_kde(strikes.T, bw_method=bandwidth / np.std(strikes, axis=0).mean())
        x, y = np.meshgrid(np.linspace(-120, 120, 200), np.linspace(-120, 120, 200))
        z = kde(np.vstack([x.ravel(), y.ravel()])).reshape(200, 200)
        z /= z.max()
        mask_pts = np.column_stack([x[z >= threshold], y[z >= threshold]])
        hull = ConvexHull(mask_pts)
        return Polygon(mask_pts[hull.vertices])
    except: return None

def evaluate_umpire_performance_final(df_subset, true_zone):
    """保留所有原始指標，並以每百球偏袒作為判定基準"""
    ump_zone = build_umpire_zone(df_subset)
    
    results = []
    for _, row in df_subset.iterrows():
        pt = Point(row["coordX"], row["coordY"])
        in_rule = true_zone.covers(pt)
        in_ump = ump_zone.covers(pt) if ump_zone is not None else False
        
        # 保留原有指標：正確性與一致性
        correct = (in_rule and row["call"] == "S") or (not in_rule and row["call"] == "B")
        consistent = (in_ump and row["call"] == "S") or (not in_ump and row["call"] == "B")
        
        results.append({
            "is_correct": correct,
            "is_consistent": consistent,
            "is_rule_strike": in_rule,
            "RE_change": row["RE_change"]
        })
    
    eval_df = pd.DataFrame(results)
    total_pitches = len(eval_df)
    
    # 指標計算
    acc = eval_df["is_correct"].mean()
    con = eval_df["is_consistent"].mean()
    strike_acc = (df_subset[eval_df["is_rule_strike"].values]["call"] == "S").mean()
    ball_acc = (df_subset[~eval_df["is_rule_strike"].values]["call"] == "B").mean()
    
    # 得分偏袒計算
    error_re_sum = eval_df[eval_df["is_correct"] == False]["RE_change"].sum()
    favor_per_100 = (error_re_sum / total_pitches) * 100 if total_pitches > 0 else 0
    
    # 根據「每百球偏袒」決定描述與顏色
    if favor_per_100 > 0.05:
        f_desc, f_col = "偏袒進攻方", "#e67e22" 
    elif favor_per_100 < -0.05:
        f_desc, f_col = "偏袒防守方", "#27ae60" 
    else:
        f_desc, f_col = "表現中立", "#7f8c8d"

    return {
        "總球數": total_pitches,
        "整體準確率": round(acc, 4),
        "一致性": round(con, 4),
        "好球準確率": round(strike_acc, 4),
        "壞球準確率": round(ball_acc, 4),
        "總得分偏袒": round(error_re_sum, 3),
        "每百球偏袒": round(favor_per_100, 3),
        "偏袒描述": f_desc,
        "偏袒顏色": f_col
    }

# --- 3. 整合繪圖函數 ---

def draw_combined_umpire_card(stats, df_subset, output_path):
    fig, (ax_plot, ax_text) = plt.subplots(1, 2, figsize=(13, 8), gridspec_kw={'width_ratios': [1.2, 1]})
    fig.patch.set_facecolor('#ffffff')

    # 左側：空間分佈 (僅標記誤判點)
    ax_plot.add_patch(Rectangle((STRIKE_ZONE_LEFT, STRIKE_ZONE_BOTTOM), 100, 100, 
                                linewidth=2, edgecolor='black', facecolor='none', label='Strike Zone (規則)', zorder=5))
    
    s_label_added = False
    b_label_added = False

    for _, row in df_subset.iterrows():
        pt = Point(row["coordX"], row["coordY"])
        is_rule_in = TRUE_ZONE_POLY.covers(pt)
        correct = (is_rule_in and row["call"] == "S") or (not is_rule_in and row["call"] == "B")
        
        if not correct:
            if row["call"] == "S": # 規則外判S
                label = "誤判好球 (好球帶外判S)" if not s_label_added else ""
                ax_plot.scatter(row["coordX"], row["coordY"], c='#2ecc71', s=60, alpha=0.7, edgecolors='k', zorder=7, label=label)
                s_label_added = True
            else: # 規則內判B
                label = "誤判壞球 (好球帶內判B)" if not b_label_added else ""
                ax_plot.scatter(row["coordX"], row["coordY"], c='#e74c3c', s=60, alpha=0.7, edgecolors='k', zorder=7, label=label)
                b_label_added = True

    uzone = build_umpire_zone(df_subset)
    if uzone:
        xh, yh = uzone.exterior.xy
        ax_plot.plot(xh, yh, color='#f39c12', linestyle='--', linewidth=5, label='Umpire Zone (裁判好球帶)', zorder=8)

    ax_plot.set_xlim(-140, 140); ax_plot.set_ylim(-140, 140)
    ax_plot.set_aspect('equal'); ax_plot.grid(True, linestyle=':', alpha=0.4)
    ax_plot.legend(loc='lower center', bbox_to_anchor=(0.5, -0.22), ncol=2, fontsize=10)
    ax_plot.set_title(f"判決誤判分佈\n({stats['打者對應']})", fontsize=15, fontweight='bold')

    # 右側：數據統計指標
    ax_text.axis('off')
    ax_text.add_patch(Rectangle((0.02, 0.02), 0.96, 0.96, transform=ax_text.transAxes, color='#f8f9fa', zorder=-1))
    
    ax_text.text(0.5, 0.92, "OVERALL UMPIRE REPORT", fontsize=12, ha='center', color='gray')
    ax_text.text(0.5, 0.86, stats['裁判'], fontsize=28, fontweight='bold', ha='center', color='#2c3e50')
    ax_text.axhline(0.82, 0.1, 0.9, color='#2c3e50', lw=2)

    y_pos = 0.74
    metrics = [
        ("整體準確率", stats['整體準確率'], "#2980b9"),
        ("一致性", stats['一致性'], "#8e44ad"),
        ("好球準確率", stats['好球準確率'], "#27ae60"),
        ("壞球準確率", stats['壞球準確率'], "#e67e22"),
    ]
    for label, val, col in metrics:
        ax_text.text(0.1, y_pos, label, fontsize=14)
        ax_text.text(0.9, y_pos, f"{val:.1%}" if not np.isnan(val) else "N/A", 
                     fontsize=18, fontweight='bold', color=col, ha='right')
        y_pos -= 0.07

    ax_text.axhline(y_pos + 0.03, 0.1, 0.9, color='#dcdde1', ls='--')

    y_pos -= 0.04
    ax_text.text(0.1, y_pos, "總得分偏袒", fontsize=14)
    ax_text.text(0.9, y_pos, f"{stats['總得分偏袒']:+.2f} Runs", fontsize=16, ha='right', color='#34495e')
    
    y_pos -= 0.09
    ax_text.text(0.1, y_pos, "每百球偏袒率", fontsize=16, fontweight='bold')
    ax_text.text(0.9, y_pos, f"{stats['每百球偏袒']:+.3f}", 
                 fontsize=24, fontweight='bold', color=stats['偏袒顏色'], ha='right')
    
    y_pos -= 0.05
    ax_text.text(0.9, y_pos, stats['偏袒描述'], fontsize=15, fontweight='bold', color=stats['偏袒顏色'], ha='right')

    ax_text.text(0.5, 0.06, f"累積判定總球數: {stats['總球數']} 球", ha='center', fontsize=11, color='gray', fontweight='bold')

    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()

# --- 4. 自動化批次處理系統 ---

def generate_umpire_report_system(df_ump):
    base_dir = "Umpire_Full_Career_Reports"
    if not os.path.exists(base_dir): os.makedirs(base_dir)

    df_ump = df_ump.dropna(subset=['umpireInChief'])
    
    # 修改：取消 Year 分組，直接按裁判分組
    for umpire, group in df_ump.groupby('umpireInChief'):
        # 資料夾直接用裁判名
        u_dir = os.path.join(base_dir, str(umpire))
        if not os.path.exists(u_dir): os.makedirs(u_dir)
        
        scenarios = [
            ("全部總結", group),
            ("面對左打", group[group['bHand'] == 'L']),
            ("面對右打", group[group['bHand'] == 'R']),
            ("左打_兩好球前", group[(group['bHand'] == 'L') & (group['strikes'] < 2)]),
            ("左打_兩好球後", group[(group['bHand'] == 'L') & (group['strikes'] == 2)]),
            ("右打_兩好球前", group[(group['bHand'] == 'R') & (group['strikes'] < 2)]),
            ("右打_兩好球後", group[(group['bHand'] == 'R') & (group['strikes'] == 2)]),
        ]

        for label, sub_df in scenarios:
            if len(sub_df) >= 12: 
                stats = evaluate_umpire_performance_final(sub_df, TRUE_ZONE_POLY)
                # 修改：年份欄位傳入 "OVERALL"
                stats.update({"年份": "OVERALL", "裁判": umpire, "打者對應": label})
                
                # 修改：檔名移除年份
                f_name = f"{label}.png"
                draw_combined_umpire_card(stats, sub_df, os.path.join(u_dir, f_name))

    print(f"✅ 全時期報表生成完成！請查看資料夾: {base_dir}")

# 執行
generate_umpire_report_system(df_ump)

✅ 全時期報表生成完成！請查看資料夾: Umpire_Full_Career_Reports


## 1. 每一年每一位裁判

In [28]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from shapely.geometry import Point, Polygon
from scipy.stats import gaussian_kde
from scipy.spatial import ConvexHull

# --- 1. 環境與好球帶設定 ---
STRIKE_ZONE_LEFT, STRIKE_ZONE_RIGHT = -50, 50
STRIKE_ZONE_BOTTOM, STRIKE_ZONE_TOP = -50, 50

TRUE_ZONE_POLY = Polygon([
    (STRIKE_ZONE_LEFT, STRIKE_ZONE_BOTTOM), (STRIKE_ZONE_RIGHT, STRIKE_ZONE_BOTTOM),
    (STRIKE_ZONE_RIGHT, STRIKE_ZONE_TOP), (STRIKE_ZONE_LEFT, STRIKE_ZONE_TOP)
])

# 中文字體設定
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'Arial Unicode MS', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False

# --- 2. 核心計算函數 ---

def build_umpire_zone(df_subset, bandwidth=12, threshold=0.35):
    """建立裁判感官好球帶 (KDE)"""
    strikes = df_subset[df_subset["call"] == "S"][["coordX", "coordY"]].dropna().values
    if len(strikes) < 4: return None 
    try:
        kde = gaussian_kde(strikes.T, bw_method=bandwidth / np.std(strikes, axis=0).mean())
        x, y = np.meshgrid(np.linspace(-120, 120, 200), np.linspace(-120, 120, 200))
        z = kde(np.vstack([x.ravel(), y.ravel()])).reshape(200, 200)
        z /= z.max()
        mask_pts = np.column_stack([x[z >= threshold], y[z >= threshold]])
        hull = ConvexHull(mask_pts)
        return Polygon(mask_pts[hull.vertices])
    except: return None

def evaluate_umpire_performance_final(df_subset, true_zone):
    """保留所有原始指標，並以每百球偏袒作為判定基準"""
    ump_zone = build_umpire_zone(df_subset)
    
    results = []
    for _, row in df_subset.iterrows():
        pt = Point(row["coordX"], row["coordY"])
        in_rule = true_zone.covers(pt)
        in_ump = ump_zone.covers(pt) if ump_zone is not None else False
        
        # 保留原有指標：正確性與一致性
        correct = (in_rule and row["call"] == "S") or (not in_rule and row["call"] == "B")
        consistent = (in_ump and row["call"] == "S") or (not in_ump and row["call"] == "B")
        
        results.append({
            "is_correct": correct,
            "is_consistent": consistent,
            "is_rule_strike": in_rule,
            "RE_change": row["RE_change"]
        })
    
    eval_df = pd.DataFrame(results)
    total_pitches = len(eval_df)
    
    # 指標計算
    acc = eval_df["is_correct"].mean()
    con = eval_df["is_consistent"].mean()
    strike_acc = (df_subset[eval_df["is_rule_strike"].values]["call"] == "S").mean()
    ball_acc = (df_subset[~eval_df["is_rule_strike"].values]["call"] == "B").mean()
    
    # 得分偏袒計算
    error_re_sum = eval_df[eval_df["is_correct"] == False]["RE_change"].sum()
    favor_per_100 = (error_re_sum / total_pitches) * 100 if total_pitches > 0 else 0
    
    # 根據「每百球偏袒」決定描述與顏色
    if favor_per_100 > 0.05:
        f_desc, f_col = "偏袒進攻方", "#e67e22" 
    elif favor_per_100 < -0.05:
        f_desc, f_col = "偏袒防守方", "#27ae60" 
    else:
        f_desc, f_col = "表現中立", "#7f8c8d"

    return {
        "總球數": total_pitches,
        "整體準確率": round(acc, 4),
        "一致性": round(con, 4),
        "好球準確率": round(strike_acc, 4),
        "壞球準確率": round(ball_acc, 4),
        "總得分偏袒": round(error_re_sum, 3),
        "每百球偏袒": round(favor_per_100, 3),
        "偏袒描述": f_desc,
        "偏袒顏色": f_col
    }

# --- 3. 整合繪圖函數 ---

def draw_combined_umpire_card(stats, df_subset, output_path):
    fig, (ax_plot, ax_text) = plt.subplots(1, 2, figsize=(13, 8), gridspec_kw={'width_ratios': [1.2, 1]})
    fig.patch.set_facecolor('#ffffff')

    # 左側：空間分佈 (僅標記誤判點)
    ax_plot.add_patch(Rectangle((STRIKE_ZONE_LEFT, STRIKE_ZONE_BOTTOM), 100, 100, 
                                linewidth=2, edgecolor='black', facecolor='none', label='Strike Zone (規則)', zorder=5))
    
    # 篩選並繪製誤判點，同時建立圖例標籤
    # 為了圖例整潔，我們手動控制標籤只出現一次
    s_label_added = False
    b_label_added = False

    for _, row in df_subset.iterrows():
        pt = Point(row["coordX"], row["coordY"])
        is_rule_in = TRUE_ZONE_POLY.covers(pt)
        correct = (is_rule_in and row["call"] == "S") or (not is_rule_in and row["call"] == "B")
        
        if not correct:
            if row["call"] == "S": # 規則外判S
                label = "誤判好球 (規則外判S)" if not s_label_added else ""
                ax_plot.scatter(row["coordX"], row["coordY"], c='#2ecc71', s=60, alpha=0.7, edgecolors='k', zorder=7, label=label)
                s_label_added = True
            else: # 規則內判B
                label = "誤判壞球 (規則內判B)" if not b_label_added else ""
                ax_plot.scatter(row["coordX"], row["coordY"], c='#e74c3c', s=60, alpha=0.7, edgecolors='k', zorder=7, label=label)
                b_label_added = True

    # 繪製 KDE 裁判好球帶
    uzone = build_umpire_zone(df_subset)
    if uzone:
        xh, yh = uzone.exterior.xy
        ax_plot.plot(xh, yh, color='#f39c12', linestyle='--', linewidth=3, label='Umpire Zone (裁判感官)', zorder=6)

    ax_plot.set_xlim(-140, 140); ax_plot.set_ylim(-140, 140)
    ax_plot.set_aspect('equal'); ax_plot.grid(True, linestyle=':', alpha=0.4)
    
    # 圖例設定
    ax_plot.legend(loc='lower center', bbox_to_anchor=(0.5, -0.22), ncol=2, fontsize=10)
    ax_plot.set_title(f"判決誤判分佈\n({stats['打者對應']})", fontsize=15, fontweight='bold')

    # 右側：數據統計指標
    ax_text.axis('off')
    ax_text.add_patch(Rectangle((0.02, 0.02), 0.96, 0.96, transform=ax_text.transAxes, color='#f8f9fa', zorder=-1))
    
    ax_text.text(0.5, 0.92, f"{stats['年份']} UMPIRE REPORT", fontsize=12, ha='center', color='gray')
    ax_text.text(0.5, 0.86, stats['裁判'], fontsize=28, fontweight='bold', ha='center', color='#2c3e50')
    ax_text.axhline(0.82, 0.1, 0.9, color='#2c3e50', lw=2)

    # 基礎指標
    y_pos = 0.74
    metrics = [
        ("整體準確率", stats['整體準確率'], "#2980b9"),
        ("一致性", stats['一致性'], "#8e44ad"),
        ("好球準確率", stats['好球準確率'], "#27ae60"),
        ("壞球準確率", stats['壞球準確率'], "#e67e22"),
    ]
    for label, val, col in metrics:
        ax_text.text(0.1, y_pos, label, fontsize=14)
        ax_text.text(0.9, y_pos, f"{val:.1%}" if not np.isnan(val) else "N/A", 
                     fontsize=18, fontweight='bold', color=col, ha='right')
        y_pos -= 0.07

    ax_text.axhline(y_pos + 0.03, 0.1, 0.9, color='#dcdde1', ls='--')

    # 得分偏袒
    y_pos -= 0.04
    ax_text.text(0.1, y_pos, "總得分偏袒", fontsize=14)
    ax_text.text(0.9, y_pos, f"{stats['總得分偏袒']:+.2f} Runs", fontsize=16, ha='right', color='#34495e')
    
    y_pos -= 0.09
    ax_text.text(0.1, y_pos, "每百球偏袒率", fontsize=16, fontweight='bold')
    ax_text.text(0.9, y_pos, f"{stats['每百球偏袒']:+.3f}", 
                 fontsize=24, fontweight='bold', color=stats['偏袒顏色'], ha='right')
    
    y_pos -= 0.05
    ax_text.text(0.9, y_pos, stats['偏袒描述'], fontsize=15, fontweight='bold', color=stats['偏袒顏色'], ha='right')

    ax_text.text(0.5, 0.06, f"總判定球數: {stats['總球數']} 球", ha='center', fontsize=11, color='gray', fontweight='bold')

    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()

# --- 4. 自動化批次處理系統 ---

def generate_umpire_report_system(df_ump):
    base_dir = "Umpire_Final_Analysis_Reports"
    if not os.path.exists(base_dir): os.makedirs(base_dir)

    df_ump = df_ump.dropna(subset=['umpireInChief'])
    
    for (year, umpire), group in df_ump.groupby(['Year', 'umpireInChief']):
        # 資料夾命名規則：年份_裁判名
        u_dir = os.path.join(base_dir, f"{year}_{umpire}")
        if not os.path.exists(u_dir): os.makedirs(u_dir)
        
        scenarios = [
            ("全部總結", group),
            ("面對左打", group[group['bHand'] == 'L']),
            ("面對右打", group[group['bHand'] == 'R']),
            ("左打_兩好球前", group[(group['bHand'] == 'L') & (group['strikes'] < 2)]),
            ("左打_兩好球後", group[(group['bHand'] == 'L') & (group['strikes'] == 2)]),
            ("右打_兩好球前", group[(group['bHand'] == 'R') & (group['strikes'] < 2)]),
            ("右打_兩好球後", group[(group['bHand'] == 'R') & (group['strikes'] == 2)]),
        ]

        for label, sub_df in scenarios:
            if len(sub_df) >= 12: 
                stats = evaluate_umpire_performance_final(sub_df, TRUE_ZONE_POLY)
                stats.update({"年份": year, "裁判": umpire, "打者對應": label})
                
                f_name = f"{year}_{label}.png"
                draw_combined_umpire_card(stats, sub_df, os.path.join(u_dir, f_name))

    print(f"✅ 報表生成完成！請查看資料夾: {base_dir}")

# 執行
generate_umpire_report_system(df_ump)

✅ 報表生成完成！請查看資料夾: Umpire_Final_Analysis_Reports
